<a href="https://colab.research.google.com/github/arturbernardo/word_vectors/blob/main/llm_vectors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install transformers torch sentencepiece

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
MODEL_NAME = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

embedding_matrix = model.get_input_embeddings().weight.detach()

print("Shape da matriz de embeddings:", tuple(embedding_matrix.shape))
print("Tamanho do vocabulário:", embedding_matrix.shape[0])
print("Dimensão dos embeddings:", embedding_matrix.shape[1])

In [ ]:
def nearest_to_vector(vec, topn=10):
    vec = vec / vec.norm()
    emb_norm = embedding_matrix / embedding_matrix.norm(dim=1, keepdim=True)
    sims = torch.mv(emb_norm, vec)

    top_ids = torch.topk(sims, k=topn).indices.tolist()

    for idx in top_ids:
        tok = tokenizer.convert_ids_to_tokens([idx])[0]
        print(tok, float(sims[idx]))

def get_embedding(token_text):
    ids = tokenizer.encode(token_text, add_special_tokens=False)

    if len(ids) != 1:
        raise ValueError(f"'{token_text}' virou vários tokens: {tokenizer.convert_ids_to_tokens(ids)}")

    return embedding_matrix[ids[0]]

In [ ]:
king = get_embedding(" king")
man = get_embedding(" man")
woman = get_embedding(" woman")

# analogia vetorial
v = king - man + woman

# buscar tokens mais próximos
nearest_to_vector(v, topn=15)

Ġking 0.7512587308883667
Ġqueen 0.7097044587135315
Ġprincess 0.6059992909431458
ĠQueen 0.5824058055877686
Ġkings 0.5818101167678833
Queen 0.5543056726455688
Ġwoman 0.5230833292007446
Ġmonarch 0.5188010334968567
Ġqueens 0.5171242952346802
ĠKing 0.5156242251396179
Ġgoddess 0.5097447633743286
Ġruler 0.5057487487792969
Ġheroine 0.5039258003234863
ĠEmpress 0.4975624084472656
Ġroyal 0.4935463070869446


In [ ]:
nearest_to_vector(king, topn=15)

Ġking 1.0
Ġkings 0.7347280979156494
ĠKing 0.6768434047698975
Ġqueen 0.6639549732208252
King 0.6070865392684937
Ġprince 0.6012954711914062
Ġmonarch 0.5855213403701782
Ġruler 0.5693265199661255
Ġemperor 0.5625233054161072
Ġroyal 0.5464542508125305
Ġprinces 0.5378309488296509
ĠQueen 0.5332906246185303
Ġkingdom 0.5309536457061768
Ġlord 0.5297277569770813
Ġthrone 0.5278258323669434
